# PretextGuard Phase 6: train the DistilBERT tactic classifier

This notebook fine-tunes DistilBERT to read an email body and give seven independent tactic probabilities
(authority, urgency, scarcity, reciprocity, social proof, liking, secrecy). All the training code lives in the
project repository (`src/models/train.py`); the notebook only prepares the Colab machine, starts it and saves the results.

**Before you run it**
1. Runtime > Change runtime type > **T4 GPU**.
2. Your Google Drive has a folder `pretextguard` that holds `tactic_data.parquet` (made on the Mac with `python -m src.models.dataset`; it holds train and validation emails only).
3. The Phase 6 code is pushed to GitHub (the notebook clones it).

Then choose Runtime > Run all. Expect about 30 to 45 minutes of GPU time (an estimate: three seeds of the main model, then three seeds of the comparison model). Cell 4 is the long one.

## 1. Check that a GPU is attached

In [ ]:
import torch

print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again from the top."
print(torch.cuda.get_device_name(0))

## 2. Get the code and pin `transformers`

The repository is public, so it is cloned without a login. `requirements.txt` in the clone says which `transformers`
version to install, so Colab and the Mac read the model files with the same library.

`torch` is **not** reinstalled: Colab already has a build that matches its GPU driver, and the default pip build of the
pinned version targets a newer CUDA that a free Colab machine may not support. The versions are printed and saved in
`tactic_run_info.json`, and the Mac check `mac_reproduces_colab` shows that both sides give the same predictions.

In [ ]:
import os, re, subprocess

REPO = "https://github.com/Nagasai-Datta/PretextGuard.git"
BRANCH = "main"
if not os.path.exists("/content/PretextGuard"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/PretextGuard"], check=True)
%cd /content/PretextGuard

pins = dict(re.findall(r"^(torch|transformers)==(\S+)", open("requirements.txt").read(), re.M))
print("requirements.txt pins:", pins)
subprocess.run(["pip", "install", "-q", "transformers==" + pins["transformers"]], check=True)
print(subprocess.run(["python", "-c", "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"],
                     capture_output=True, text=True).stdout)
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

## 3. Mount Google Drive and check the data

Colab asks for permission to read your Drive. The check prints how many emails of each kind the file holds and stops if a
test row is in it: the test split must never reach the training code (the test labels are used once, in Phase 13).

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
DRIVE_FOLDER = "/content/drive/MyDrive/pretextguard"
DATA = DRIVE_FOLDER + "/tactic_data.parquet"
OUT = "/content/phase6_outputs"

table = pd.read_parquet(DATA)
print(table.groupby(["origin", "split"]).size())
assert set(table["split"]) == {"train", "validation"}, "the file must hold train and validation rows only"

## 4. Train

`train.py` trains two conditions, each with seeds 42, 43 and 44:
- **mix**: the real and the synthetic training emails. This one becomes the model.
- **real_only**: the real training emails alone, to measure whether the synthetic emails help.

Every epoch prints the training loss, the validation loss and the validation macro-F1 on real emails. Training stops after
3 epochs without improvement and keeps the best epoch. Hugging Face may print a warning that the classifier layer is newly
initialised: that is expected, because that layer is what we train.

In [ ]:
!python -u -m src.models.train --data "{DATA}" --out "{OUT}"

## 5. Look at the results

The seed table says which seed was chosen. The plot shows the chosen model's training and validation loss: if the training
loss keeps falling while the validation loss rises, the model is memorising (overfitting).

In [ ]:
import json
import matplotlib.pyplot as plt

seeds = pd.read_csv(OUT + "/tactic_seed_summary.csv")
print(seeds.to_string(index=False))
info = json.load(open(OUT + "/tactic_run_info.json"))
for condition, details in info["conditions"].items():
    print(condition, "thresholds:", details["thresholds"])

log = pd.read_csv(OUT + "/tactic_training_log.csv")
chosen = seeds[(seeds.condition == "mix") & seeds.chosen].iloc[0]
run = log[(log.condition == "mix") & (log.seed == chosen.seed)]
plt.plot(run.epoch, run.train_loss, label="training loss")
plt.plot(run.epoch, run.val_loss, label="validation loss")
plt.axvline(chosen.best_epoch, color="grey", linestyle=":", label="chosen epoch")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("mix, seed %d" % chosen.seed)
plt.show()

## 6. Save everything to Drive

The zip holds the model folder and four small files. Download `phase6_outputs.zip` from the `pretextguard` folder in Google
Drive (browser), then follow `notebooks/README.md`, section "Back on the Mac".

In [ ]:
import shutil

zip_path = shutil.make_archive(DRIVE_FOLDER + "/phase6_outputs", "zip", OUT)
print(zip_path, round(os.path.getsize(zip_path) / 1e6), "MB")